In [26]:
# %% Cell 1 — Load the merged dataset from notebook 1
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/merged.parquet')

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print()
print(df.head())

Loaded 5665 rows
Columns: ['date', 'rides', 'temp_mean', 'temp_max', 'temp_min', 'precipitation', 'windspeed', 'weather_code', 'weather_desc', 'is_holiday', 'year', 'month', 'day_of_week', 'is_weekend', 'season', 'season_name', 'is_anomaly', 'anomaly_reason']

        date  rides  temp_mean  temp_max  temp_min  precipitation  windspeed  \
0 2010-07-30   6897       17.6      22.0      12.6            0.0       16.5   
1 2010-07-31   5564       19.1      22.8      16.8            0.1       18.2   
2 2010-08-01   4303       17.6      21.8      13.1            0.0       12.3   
3 2010-08-02   6642       17.4      21.0      14.9            0.2       13.7   
4 2010-08-03   7966       17.5      21.7      12.4            0.0       17.7   

   weather_code   weather_desc  is_holiday  year  month  day_of_week  \
0             3  Partly cloudy           0  2010      7            4   
1            51        Drizzle           0  2010      7            5   
2             3  Partly cloudy           0

In [27]:
# %% Cell 2 — Check the data looks correct
print("=== DATA CHECK ===")
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Data types:")
print(df.dtypes)
print()
print("Basic statistics:")
print(df[['rides', 'temp_mean', 'temp_max',
          'temp_min', 'precipitation', 'windspeed']].describe().round(2))

=== DATA CHECK ===
Shape: (5665, 18)
Date range: 2010-07-30 to 2026-01-31

Missing values per column:
date              0
rides             0
temp_mean         0
temp_max          0
temp_min          0
precipitation     0
windspeed         0
weather_code      0
weather_desc      0
is_holiday        0
year              0
month             0
day_of_week       0
is_weekend        0
season            0
season_name       0
is_anomaly        0
anomaly_reason    0
dtype: int64

Data types:
date              datetime64[us]
rides                      int64
temp_mean                float64
temp_max                 float64
temp_min                 float64
precipitation            float64
windspeed                float64
weather_code               int64
weather_desc                 str
is_holiday                 int64
year                       int32
month                      int32
day_of_week                int32
is_weekend                 int64
season                     int64
season_name      

In [28]:
# %% Cell 3 — Remove the 0 ride anomaly days
# The two Queen Elizabeth mourning days (Sept 10-11 2022)
# still in the parquet file with rides = 0
# We remove them here before any analysis

print(f"Rows before: {len(df)}")
print()
print("Zero ride days in dataset:")
print(df[df['rides'] == 0][['date', 'rides', 'is_anomaly', 'anomaly_reason']])

df = df[df['rides'] > 0].reset_index(drop=True)

print(f"\nRows after removal: {len(df)}")
print(f"Zero ride days remaining: {len(df[df['rides'] == 0])}")

Rows before: 5665

Zero ride days in dataset:
           date  rides  is_anomaly                          anomaly_reason
4425 2022-09-10      0           1  National mourning - Queen Elizabeth II
4426 2022-09-11      0           1  National mourning - Queen Elizabeth II

Rows after removal: 5663
Zero ride days remaining: 0


In [29]:
# %% Cell 4 — Check for any remaining weird values
print("=== CHECKING FOR WEIRD VALUES ===")

# Negative rides (impossible)
neg_rides = df[df['rides'] < 0]
print(f"Negative ride counts: {len(neg_rides)}")

# Unrealistic temperatures for London
# London never goes below -15 or above 40
weird_temp = df[(df['temp_mean'] < -15) | (df['temp_mean'] > 40)]
print(f"Unrealistic temperatures: {len(weird_temp)}")

# Negative precipitation (impossible)
neg_precip = df[df['precipitation'] < 0]
print(f"Negative precipitation values: {len(neg_precip)}")

print(f"\nMin rides: {df['rides'].min()}")
print(f"Max rides: {df['rides'].max()}")
print(f"Min temperature: {df['temp_mean'].min()}°C")
print(f"Max temperature: {df['temp_mean'].max()}°C")

print()
if len(neg_rides) == 0 and len(weird_temp) == 0 and len(neg_precip) == 0:
    print("All checks passed! No weird values found.")
else:
    print("Some issues found, check above.")

=== CHECKING FOR WEIRD VALUES ===
Negative ride counts: 0
Unrealistic temperatures: 0
Negative precipitation values: 0

Min rides: 188
Max rides: 73094
Min temperature: -6.5°C
Max temperature: 29.1°C

All checks passed! No weird values found.


In [30]:
# %% Cell 5 — Add a rain flag
# Simple yes/no whether it rained that day
# Anything above 1mm of precipitation counts as a rainy day
# This is useful as a clean variable for the model and analysis

df['is_rainy'] = (df['precipitation'] > 1).astype(int)

print("Rainy vs dry days:")
print(df['is_rainy'].value_counts())
print()
print(f"Rainy days: {df['is_rainy'].sum()} ({df['is_rainy'].mean()*100:.1f}%)")
print(f"Dry days:   {(df['is_rainy']==0).sum()} ({(df['is_rainy']==0).mean()*100:.1f}%)")
print()
print("Average rides on rainy days:  ", df[df['is_rainy']==1]['rides'].mean().round(0))
print("Average rides on dry days:    ", df[df['is_rainy']==0]['rides'].mean().round(0))

Rainy vs dry days:
is_rainy
0    3700
1    1963
Name: count, dtype: int64

Rainy days: 1963 (34.7%)
Dry days:   3700 (65.3%)

Average rides on rainy days:   22564.0
Average rides on dry days:     28123.0


In [31]:
# %% Cell 6 — Add day type column
# Combines holiday, weekend and weekday into one single label
# This makes it very easy to compare the three groups
# directly in graphs and in the model

def day_type(row):
    if row['is_holiday'] == 1:
        return 'Holiday'
    elif row['is_weekend'] == 1:
        return 'Weekend'
    else:
        return 'Weekday'

df['day_type'] = df.apply(day_type, axis=1)

print("Day type distribution:")
print(df['day_type'].value_counts())
print()
print("Average rides by day type:")
print(df.groupby('day_type')['rides'].mean().round(0).sort_values(ascending=False))

Day type distribution:
day_type
Weekday    3917
Weekend    1617
Holiday     129
Name: count, dtype: int64

Average rides by day type:
day_type
Weekday    27706.0
Weekend    23026.0
Holiday    20111.0
Name: rides, dtype: float64


In [32]:
# %% Cell 7 — Add yearly average context
# London cycling grew a lot between 2010 and 2026
# A day with 20,000 rides in 2011 is a big day
# but 20,000 rides in 2020 is a quiet day
# This column shows rides relative to that year's average
# So 1.0 = exactly average for that year
#    1.2 = 20% above average for that year
#    0.8 = 20% below average for that year
# This helps separate weather/holiday effects 
# from the long term growth trend

yearly_avg = df.groupby('year')['rides'].transform('mean')
df['rides_vs_yearly_avg'] = (df['rides'] / yearly_avg).round(3)

print("Yearly average rides (shows growth over time):")
print(df.groupby('year')['rides'].mean().round(0).to_string())
print()
print("Sample of rides_vs_yearly_avg column:")
print(df[['date', 'rides', 'rides_vs_yearly_avg']].head(10))

Yearly average rides (shows growth over time):
year
2010    14070.0
2011    19568.0
2012    26009.0
2013    22042.0
2014    27463.0
2015    27046.0
2016    28152.0
2017    28619.0
2018    28952.0
2019    28562.0
2020    28509.0
2021    29976.0
2022    31697.0
2023    23373.0
2024    23957.0
2025    24840.0
2026    18611.0

Sample of rides_vs_yearly_avg column:
        date  rides  rides_vs_yearly_avg
0 2010-07-30   6897                0.490
1 2010-07-31   5564                0.395
2 2010-08-01   4303                0.306
3 2010-08-02   6642                0.472
4 2010-08-03   7966                0.566
5 2010-08-04   7893                0.561
6 2010-08-05   8724                0.620
7 2010-08-06   9797                0.696
8 2010-08-07   6631                0.471
9 2010-08-08   7864                0.559


In [33]:
# %% Cell 8 — Final summary
print("=== FINAL CLEAN DATASET SUMMARY ===")
print(f"Total rows (days):       {len(df)}")
print(f"Total columns:           {len(df.columns)}")
print(f"Date range:              {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Missing values:          {df.isnull().sum().sum()}")
print()
print("--- Columns in dataset ---")
for col in df.columns:
    print(f"  {col}")
print()
print("--- Average rides by day type ---")
print(df.groupby('day_type')['rides'].mean().round(0).sort_values(ascending=False))
print()
print("--- Average rides by season ---")
print(df.groupby('season_name')['rides'].mean().round(0).sort_values(ascending=False))
print()
print("--- Average rides by rain ---")
print(df.groupby('is_rainy')['rides'].mean().round(0))
print()
print("--- Temperature range ---")
print(f"Coldest day:  {df['temp_mean'].min()}°C on {df.loc[df['temp_mean'].idxmin(), 'date'].date()}")
print(f"Hottest day:  {df['temp_mean'].max()}°C on {df.loc[df['temp_mean'].idxmax(), 'date'].date()}")

=== FINAL CLEAN DATASET SUMMARY ===
Total rows (days):       5663
Total columns:           21
Date range:              2010-07-30 to 2026-01-31
Missing values:          0

--- Columns in dataset ---
  date
  rides
  temp_mean
  temp_max
  temp_min
  precipitation
  windspeed
  weather_code
  weather_desc
  is_holiday
  year
  month
  day_of_week
  is_weekend
  season
  season_name
  is_anomaly
  anomaly_reason
  is_rainy
  day_type
  rides_vs_yearly_avg

--- Average rides by day type ---
day_type
Weekday    27706.0
Weekend    23026.0
Holiday    20111.0
Name: rides, dtype: float64

--- Average rides by season ---
season_name
Summer    32768.0
Autumn    27040.0
Spring    26359.0
Winter    18613.0
Name: rides, dtype: float64

--- Average rides by rain ---
is_rainy
0    28123.0
1    22564.0
Name: rides, dtype: float64

--- Temperature range ---
Coldest day:  -6.5°C on 2010-12-03
Hottest day:  29.1°C on 2022-07-19


In [34]:
# Drop anomaly columns, no longer needed
df = df.drop(columns=['is_anomaly', 'anomaly_reason'])
print("Dropped anomaly columns")
print(f"Columns now: {len(df.columns)}")

Dropped anomaly columns
Columns now: 19


In [35]:
# %% Cell 9 — Save the clean dataset
df.to_parquet('../data/processed/cleaned.parquet', index=False)
print("Saved: data/processed/cleaned.parquet")
print()
print("Notebook 1 produced: merged.parquet  (raw merge of all sources)")
print("Notebook 2 produced: cleaned.parquet (ready for analysis)")
print()
print("All future notebooks load cleaned.parquet")

Saved: data/processed/cleaned.parquet

Notebook 1 produced: merged.parquet  (raw merge of all sources)
Notebook 2 produced: cleaned.parquet (ready for analysis)

All future notebooks load cleaned.parquet
